# Ejercicio 08: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../data/beir_datasets\\scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [12]:
import numpy as np
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Preparar el corpus
# Extraemos los IDs y concatenamos 'title' y 'text' para tener el contenido completo
corpus_ids = list(corpus.keys())
corpus_texts = [
    (corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")).lower()
    for doc_id in corpus_ids
]

# Tokenización simple por espacios (suficiente para un baseline rápido)
print("Tokenizando el corpus...")
tokenized_corpus = [doc.split() for doc in corpus_texts]

# 2. Construir el modelo BM25
print("Construyendo el índice BM25...")
bm25 = BM25Okapi(tokenized_corpus)

# 3. Realizar el retrieval para todas las queries
print("Calculando scores para las queries...")
results = {}
top_k = 100

for qid, query_text in queries.items():
    # Tokenizar la query
    tokenized_query = query_text.lower().split()

    # Obtener scores de todos los documentos para esta query
    doc_scores = bm25.get_scores(tokenized_query)

    # Obtener los índices de los top_k documentos con mayor score
    top_indices = np.argsort(doc_scores)[::-1][:top_k]

    results[qid] = {corpus_ids[idx]: float(doc_scores[idx]) for idx in top_indices}

# 4. Evaluación de métricas
print("Evaluando métricas...")
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels,
    results,
    k_values=[10]
)

print("\n--- Resultados Baseline (BM25) ---")
print(f"Recall@10: {recall['Recall@10']:.4f}")
print(f"nDCG@10:  {ndcg['NDCG@10']:.4f}")

Tokenizando el corpus...
Construyendo el índice BM25...
Calculando scores para las queries...
Evaluando métricas...

--- Resultados Baseline (BM25) ---
Recall@10: 0.6862
nDCG@10:  0.5597


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [15]:
from sentence_transformers import CrossEncoder
import pandas as pd
from tqdm import tqdm

# Modelo Cross-Encoder
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

TOP_K_BM25 = 100
TOP_K_FINAL = 10

reranked_results = {}
rank_changes = []

sample_queries = dict(list(queries.items())[:20])#para probar 20 aaa

for qid, query in tqdm(sample_queries.items()):

    # Top 100 candidatos de BM25
    bm25_candidates = sorted(
        results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:TOP_K_BM25]

    candidate_doc_ids = [doc_id for doc_id, _ in bm25_candidates]

    pairs = [
        (
            query,
            corpus[doc_id]["title"] + " " + corpus[doc_id]["text"]
        )
        for doc_id in candidate_doc_ids
    ]

    # Scores del Cross-Encoder
    ce_scores = cross_encoder.predict(
        pairs,
        batch_size=16,
        show_progress_bar=True
    )

    # Re-ranking
    reranked = sorted(
        zip(candidate_doc_ids, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_top10 = reranked[:TOP_K_FINAL]

    reranked_results[qid] = {
        doc_id: float(score)
        for doc_id, score in reranked_top10
    }

    # Comparación de posiciones
    bm25_top10 = [doc_id for doc_id, _ in bm25_candidates[:10]]
    ce_top10 = [doc_id for doc_id, _ in reranked_top10]

    for doc_id in set(bm25_top10 + ce_top10):

        pos_bm25 = (
            bm25_top10.index(doc_id) + 1
            if doc_id in bm25_top10 else None
        )

        pos_ce = (
            ce_top10.index(doc_id) + 1
            if doc_id in ce_top10 else None
        )

        if pos_bm25 != pos_ce:
            rank_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": pos_bm25,
                "cross_encoder_rank": pos_ce,
            })

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from beir.retrieval.evaluation import EvaluateRetrieval
from tqdm import tqdm
import pandas as pd
import numpy as np

TOP_K_BM25 = 50
TOP_K_FINAL = 10

# ============================================================
# Preparación de textos
# ============================================================

doc_texts = {
    doc_id: (
            corpus[doc_id].get("title", "") + " " +
            corpus[doc_id].get("text", "")
    ).lower()
    for doc_id in corpus_ids
}

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

doc_matrix = tfidf_vectorizer.fit_transform(
    [doc_texts[doc_id] for doc_id in corpus_ids]
)

doc_id_to_idx = {
    doc_id: idx
    for idx, doc_id in enumerate(corpus_ids)
}

query_vectors = {
    qid: tfidf_vectorizer.transform([query.lower()])
    for qid, query in queries.items()
}


# ============================================================
# Construcción de Features
# ============================================================

def build_ltr_features(
        qid,
        query_text,
        doc_id,
        bm25_score,
        bm25_rank
):
    query_terms = query_text.lower().split()
    doc_terms = doc_texts[doc_id].split()

    query_len = len(query_terms)
    doc_len = len(doc_terms)

    overlap = len(set(query_terms) & set(doc_terms))

    overlap_ratio = (
        overlap / query_len
        if query_len > 0 else 0
    )

    tfidf_cos = cosine_similarity(
        query_vectors[qid],
        doc_matrix[doc_id_to_idx[doc_id]]
    )[0][0]

    return [
        bm25_score,
        bm25_rank,
        query_len,
        doc_len,
        overlap,
        overlap_ratio,
        tfidf_cos
    ]


# ============================================================
# Dataset de entrenamiento
# ============================================================

X = []
y = []

for qid, query_text in tqdm(
        queries.items(),
        desc="Construyendo dataset LTR"
):

    bm25_candidates = sorted(
        results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:TOP_K_BM25]

    for rank, (doc_id, bm25_score) in enumerate(
            bm25_candidates,
            start=1
    ):
        X.append(
            build_ltr_features(
                qid,
                query_text,
                doc_id,
                bm25_score,
                rank
            )
        )

        y.append(
            qrels.get(qid, {}).get(doc_id, 0)
        )

X = np.array(X)
y = np.array(y)

# ============================================================
# Entrenamiento LTR
# ============================================================

ltr_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

ltr_model.fit(X, y)

# ============================================================
# Re-ranking
# ============================================================

ltr_results = {}
ltr_rank_changes = []

for qid, query_text in tqdm(
        queries.items(),
        desc="Re-ranking LTR"
):

    bm25_candidates = sorted(
        results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:TOP_K_BM25]

    candidate_doc_ids = [
        doc_id
        for doc_id, _ in bm25_candidates
    ]

    X_candidates = np.array([
        build_ltr_features(
            qid,
            query_text,
            doc_id,
            bm25_score,
            rank
        )
        for rank, (doc_id, bm25_score)
        in enumerate(bm25_candidates, start=1)
    ])

    ltr_scores = ltr_model.predict(
        X_candidates
    )

    reranked = sorted(
        zip(candidate_doc_ids, ltr_scores),
        key=lambda x: x[1],
        reverse=True
    )

    top10_ltr = reranked[:TOP_K_FINAL]

    ltr_results[qid] = {
        doc_id: float(score)
        for doc_id, score in top10_ltr
    }

    bm25_top10 = [
        doc_id
        for doc_id, _
        in bm25_candidates[:TOP_K_FINAL]
    ]

    ltr_top10 = [
        doc_id
        for doc_id, _
        in top10_ltr
    ]

    for doc_id in set(
            bm25_top10 + ltr_top10
    ):

        pos_bm25 = (
            bm25_top10.index(doc_id) + 1
            if doc_id in bm25_top10 else None
        )

        pos_ltr = (
            ltr_top10.index(doc_id) + 1
            if doc_id in ltr_top10 else None
        )

        if pos_bm25 != pos_ltr:
            ltr_rank_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": pos_bm25,
                "ltr_rank": pos_ltr,
                "status": (
                    "entra" if pos_bm25 is None
                    else "sale" if pos_ltr is None
                    else "sube" if pos_ltr < pos_bm25
                    else "baja"
                )
            })

# ============================================================
# Evaluación
# ============================================================

ndcg_ltr, map_ltr, recall_ltr, precision_ltr = (
    EvaluateRetrieval.evaluate(
        qrels,
        ltr_results,
        k_values=[10]
    )
)

print("\n=== RESULTADOS LTR ===")
print(f"Recall@10 : {recall_ltr['Recall@10']:.4f}")
print(f"nDCG@10   : {ndcg_ltr['NDCG@10']:.4f}")

df_ltr_rank_changes = pd.DataFrame(
    ltr_rank_changes
)

print(
    f"\nTotal cambios top10: "
    f"{len(df_ltr_rank_changes)}"
)

display(
    df_ltr_rank_changes.head(20)
)

Re-ranking LTR: 100%|██████████| 300/300 [00:18<00:00, 16.07it/s]


=== RESULTADOS LTR ===
Recall@10 : 0.7652
nDCG@10   : 0.7125

Total cambios top10: 2690


,query_id,doc_id,bm25_rank,ltr_rank,status
0,1,43385013,3.0,4.0,baja
1,1,45638119,7.0,8.0,baja
2,1,18953920,5.0,6.0,baja
3,1,17388232,6.0,3.0,sube
4,1,25298276,10.0,9.0,sube
5,1,24998637,8.0,7.0,sube
6,1,17518195,9.0,10.0,baja
7,1,13231899,4.0,5.0,baja
8,3,3672261,1.0,3.0,baja
9,3,14717500,2.0,1.0,sube


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [18]:
from beir.retrieval.evaluation import EvaluateRetrieval

K_VALUES = [10]

# ==========================
# BM25
# ==========================
ndcg_bm25, map_bm25, recall_bm25, precision_bm25 = (
    EvaluateRetrieval.evaluate(
        qrels,
        results,
        K_VALUES
    )
)

# ==========================
# Cross Encoder
# ==========================
ndcg_ce, map_ce, recall_ce, precision_ce = (
    EvaluateRetrieval.evaluate(
        qrels,
        ce_results,
        K_VALUES
    )
)

# ==========================
# LTR
# ==========================
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = (
    EvaluateRetrieval.evaluate(
        qrels,
        ltr_results,
        K_VALUES
    )
)

print("=== BM25 ===")
print(f"nDCG@10   : {ndcg_bm25['NDCG@10']:.4f}")
print(f"MAP@10    : {map_bm25['MAP@10']:.4f}")
print(f"Recall@10 : {recall_bm25['Recall@10']:.4f}")

print("\n=== Cross Encoder ===")
print(f"nDCG@10   : {ndcg_ce['NDCG@10']:.4f}")
print(f"MAP@10    : {map_ce['MAP@10']:.4f}")
print(f"Recall@10 : {recall_ce['Recall@10']:.4f}")

print("\n=== LTR ===")
print(f"nDCG@10   : {ndcg_ltr['NDCG@10']:.4f}")
print(f"MAP@10    : {map_ltr['MAP@10']:.4f}")
print(f"Recall@10 : {recall_ltr['Recall@10']:.4f}")

=== BM25 ===
nDCG@10   : 0.5597
MAP@10    : 0.5147
Recall@10 : 0.6862

=== Cross Encoder ===
nDCG@10   : 0.5741
MAP@10    : 0.5513
Recall@10 : 0.6253

=== LTR ===
nDCG@10   : 0.7125
MAP@10    : 0.6904
Recall@10 : 0.7652


In [19]:
import pandas as pd

comparison = pd.DataFrame({
    "Método": ["BM25", "Cross-Encoder", "LTR"],
    "nDCG@10": [
        ndcg_bm25["NDCG@10"],
        ndcg_ce["NDCG@10"],
        ndcg_ltr["NDCG@10"]
    ],
    "MAP@10": [
        map_bm25["MAP@10"],
        map_ce["MAP@10"],
        map_ltr["MAP@10"]
    ],
    "Recall@10": [
        recall_bm25["Recall@10"],
        recall_ce["Recall@10"],
        recall_ltr["Recall@10"]
    ]
})

comparison = comparison.round(4)

display(comparison)

,Método,nDCG@10,MAP@10,Recall@10
0,BM25,0.5597,0.5147,0.6862
1,Cross-Encoder,0.5741,0.5513,0.6253
2,LTR,0.7125,0.6904,0.7652
